# Module 2: Research Agent for Commodity Analysis

## Learning Objectives

By completing this notebook, you will:
1. Build autonomous research agents using LangChain and ReAct patterns
2. Implement multi-step reasoning for complex commodity questions
3. Combine RAG retrieval with agentic tool use (calculator, search, data access)
4. Create research workflows that iterate and refine answers
5. Handle ambiguous queries with clarification and decomposition

## Prerequisites

- Completed Module 2, Notebook 1 (EIA Knowledge Base)
- Understanding of RAG systems
- Familiarity with LangChain or agent frameworks
- OpenAI or Anthropic API access

## Estimated Time: 90-110 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completed Module 2, Notebook 1 (EIA Knowledge Base)",
    "Understanding of RAG systems",
    "Familiarity with LangChain or agent frameworks",
    "OpenAI or Anthropic API access"
])

## 1. Understanding Research Agents

**Problem with Simple RAG:**
- Can only answer questions directly in documents
- No reasoning or calculation ability
- Cannot break down complex questions
- No iteration or refinement

**Solution: Research Agents**
- **ReAct Pattern**: Reason + Act in loops
- **Tools**: RAG search, calculator, API calls, data analysis
- **Planning**: Break questions into sub-questions
- **Memory**: Track progress and intermediate results

**Example:**
```
Question: "What was the percent change in crude oil inventories over the last month?"

Agent reasoning:
1. Need to find latest inventory level -> Use RAG tool
2. Need inventory from 1 month ago -> Use RAG tool with date filter
3. Calculate percent change -> Use calculator tool
4. Format and return answer
```

In [ ]:
# Purpose: Setup imports and configuration for research agent
# Key Concept: Agent frameworks coordinate LLMs with tools

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# LangChain imports
from langchain.agents import AgentExecutor, create_react_agent, Tool
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory
from langchain.tools import BaseTool
from pydantic import BaseModel, Field

# Vector database (from previous notebook)
import chromadb
from sentence_transformers import SentenceTransformer

# Configuration
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize LLM
if OPENAI_API_KEY:
    llm = ChatOpenAI(model="gpt-4", temperature=0.0, api_key=OPENAI_API_KEY)
    print("✅ Using OpenAI GPT-4")
elif ANTHROPIC_API_KEY:
    llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0.0, api_key=ANTHROPIC_API_KEY)
    print("✅ Using Claude 3.5 Sonnet")
else:
    llm = None
    print("⚠️  No LLM configured")

# Data directories
DATA_DIR = Path('data/eia_reports')
CHROMA_DIR = Path('data/chroma_db')

# Load vector database from previous notebook
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    collection = chroma_client.get_collection(name="eia_petroleum_reports")
    print(f"✅ Loaded EIA vector database ({collection.count()} documents)")
except:
    collection = None
    print("⚠️  No vector database found. Run Module 2, Notebook 1 first.")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\n✅ Setup complete")

## 2. Building Custom Tools

Research agents need tools to interact with the world. We'll create:

1. **RAG Search Tool** - Query the EIA knowledge base
2. **Calculator Tool** - Perform mathematical calculations
3. **Data Extractor Tool** - Parse numbers from text
4. **Date Tool** - Handle date calculations

Each tool has:
- Name and description (agent uses these to decide when to use it)
- Input schema
- Execute function

In [ ]:
# Purpose: Create RAG search tool for agent
# Key Concept: Tools are functions the agent can call with specific inputs

class RAGSearchInput(BaseModel):
    """Input schema for RAG search tool."""
    query: str = Field(description="Natural language query about commodity markets")
    date_filter: Optional[Dict[str, str]] = Field(
        default=None,
        description="Optional date filter with 'start' and/or 'end' keys (YYYY-MM-DD format)"
    )
    top_k: int = Field(default=3, description="Number of results to return")

class RAGSearchTool(BaseTool):
    """Tool for searching EIA petroleum reports."""
    
    name = "eia_report_search"
    description = """
    Search EIA Weekly Petroleum Status Reports for information about crude oil, gasoline,
    distillates, inventories, production, refining, and demand. Returns relevant excerpts
    from actual reports with dates.
    
    Use this when you need factual data from official EIA reports.
    
    Input: query (string), optional date_filter (dict), optional top_k (int)
    Output: List of relevant report excerpts with dates
    """
    args_schema: type[BaseModel] = RAGSearchInput
    
    def _run(self, query: str, date_filter: Optional[Dict[str, str]] = None, top_k: int = 3) -> str:
        """Execute RAG search."""
        if not collection:
            return "Error: Vector database not available"
        
        # Generate query embedding
        query_embedding = embedding_model.encode([query])[0]
        
        # Build filter
        where_filter = None
        if date_filter:
            where_filter = {}
            if 'start' in date_filter:
                where_filter['report_date'] = {'$gte': date_filter['start']}
            if 'end' in date_filter:
                if 'report_date' not in where_filter:
                    where_filter['report_date'] = {}
                where_filter['report_date']['$lte'] = date_filter['end']
        
        # Search
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            where=where_filter if where_filter else None
        )
        
        if not results['ids'] or len(results['ids'][0]) == 0:
            return "No results found"
        
        # Format results
        formatted = []
        for i in range(len(results['ids'][0])):
            text = results['documents'][0][i]
            metadata = results['metadatas'][0][i]
            date = metadata.get('report_date', 'unknown')
            formatted.append(f"[{date}] {text}")
        
        return "\n\n".join(formatted)
    
    async def _arun(self, *args, **kwargs):
        """Async version."""
        return self._run(*args, **kwargs)

# Test the tool
if collection:
    rag_tool = RAGSearchTool()
    test_result = rag_tool._run("crude oil inventories", top_k=2)
    print("RAG Tool Test:")
    print(test_result[:300] + "...")
else:
    print("⚠️  Cannot test tool without vector database")

In [ ]:
# Purpose: Create calculator and data extraction tools
# Key Concept: Simple tools handle specific tasks agents can't do alone

class CalculatorInput(BaseModel):
    """Input for calculator tool."""
    expression: str = Field(description="Mathematical expression to evaluate (Python syntax)")

class CalculatorTool(BaseTool):
    """Tool for mathematical calculations."""
    
    name = "calculator"
    description = """
    Perform mathematical calculations. Supports basic arithmetic, percentages,
    and Python math functions.
    
    Examples:
    - "(450 - 440) / 440 * 100" -> 2.27 (percent change)
    - "sum([1, 2, 3, 4, 5])" -> 15
    - "max(100, 200, 150)" -> 200
    
    Input: expression (string)
    Output: numerical result
    """
    args_schema: type[BaseModel] = CalculatorInput
    
    def _run(self, expression: str) -> str:
        """Evaluate mathematical expression."""
        try:
            # Safe eval with limited scope
            allowed_names = {
                'abs': abs, 'min': min, 'max': max, 'sum': sum,
                'round': round, 'len': len, 'pow': pow
            }
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            return str(result)
        except Exception as e:
            return f"Error evaluating expression: {str(e)}"
    
    async def _arun(self, *args, **kwargs):
        return self._run(*args, **kwargs)

class DataExtractorInput(BaseModel):
    """Input for data extractor."""
    text: str = Field(description="Text containing numerical data")
    data_type: str = Field(description="Type of data to extract: 'number', 'percentage', 'date'")

class DataExtractorTool(BaseTool):
    """Tool for extracting structured data from text."""
    
    name = "data_extractor"
    description = """
    Extract numbers, percentages, or dates from text.
    Useful for parsing report text to get specific values.
    
    Input: text (string), data_type ('number', 'percentage', or 'date')
    Output: extracted values
    """
    args_schema: type[BaseModel] = DataExtractorInput
    
    def _run(self, text: str, data_type: str) -> str:
        """Extract data from text."""
        import re
        
        if data_type == 'number':
            # Extract numbers with optional commas and decimals
            pattern = r'\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b'
            matches = re.findall(pattern, text)
            # Remove commas and convert to float
            numbers = [float(m.replace(',', '')) for m in matches]
            return json.dumps(numbers)
        
        elif data_type == 'percentage':
            # Extract percentages
            pattern = r'(-?\d+(?:\.\d+)?)\s*%'
            matches = re.findall(pattern, text)
            percentages = [float(m) for m in matches]
            return json.dumps(percentages)
        
        elif data_type == 'date':
            # Extract dates (simple pattern)
            pattern = r'\d{4}-\d{2}-\d{2}'
            matches = re.findall(pattern, text)
            return json.dumps(matches)
        
        else:
            return f"Unknown data_type: {data_type}"
    
    async def _arun(self, *args, **kwargs):
        return self._run(*args, **kwargs)

# Test tools
calc_tool = CalculatorTool()
extractor_tool = DataExtractorTool()

print("Calculator Tool Test:")
print(f"  (450 - 440) / 440 * 100 = {calc_tool._run('(450 - 440) / 440 * 100')}")

print("\nData Extractor Tool Test:")
sample_text = "Inventories rose by 2.5 million barrels to 445.3 million barrels, up 3.2% from last week."
print(f"  Numbers: {extractor_tool._run(sample_text, 'number')}")
print(f"  Percentages: {extractor_tool._run(sample_text, 'percentage')}")

print("\n✅ All tools ready")

## 3. Creating the Research Agent

Now we assemble the agent using LangChain's ReAct pattern.

**ReAct Loop:**
1. **Thought**: Agent reasons about the question
2. **Action**: Agent selects a tool and input
3. **Observation**: Tool returns result
4. Repeat until agent has enough information
5. **Final Answer**: Agent provides answer

The agent prompt tells it:
- What tools are available
- When to use each tool
- How to format actions
- When it has enough info to answer

In [ ]:
# Purpose: Create ReAct agent with tools
# Key Concept: Agent orchestrates tools to solve complex problems

# Define tools
tools = [
    rag_tool if collection else None,
    calc_tool,
    extractor_tool
]
tools = [t for t in tools if t is not None]  # Remove None values

# Create ReAct prompt
react_prompt = PromptTemplate.from_template("""
You are a commodity market research analyst with access to EIA petroleum reports and analysis tools.

Answer the following question by using the available tools. Think step by step.

You have access to these tools:
{tools}

Tool names: {tool_names}

Use this format:

Question: the input question
Thought: think about what to do
Action: tool name
Action Input: the input to the tool
Observation: the result of the action
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

# Create agent
if llm and tools:
    agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=react_prompt
    )
    
    # Create executor
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        max_iterations=10,
        handle_parsing_errors=True
    )
    
    print("✅ Research agent created")
    print(f"   Tools: {[t.name for t in tools]}")
    print(f"   Max iterations: 10")
else:
    agent_executor = None
    print("⚠️  Cannot create agent without LLM and tools")

In [ ]:
# Purpose: Test the research agent with a simple question
# Key Concept: Agent autonomously decides which tools to use

if agent_executor:
    print("Testing Research Agent\n")
    print("="*70)
    
    test_question = "What were the recent crude oil inventory levels according to EIA reports?"
    
    print(f"Question: {test_question}\n")
    print("Agent Execution:")
    print("-"*70)
    
    try:
        result = agent_executor.invoke({"input": test_question})
        
        print("\n" + "="*70)
        print("Final Answer:")
        print("="*70)
        print(result['output'])
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
else:
    print("⚠️  Agent not available")

### 💡 Exercise 3.1: Multi-Step Reasoning

**Task:** Test the agent with a question requiring multiple steps and calculations.

**Example Questions:**
1. "What is the percent change in crude oil inventories between the oldest and newest reports?"
2. "Compare gasoline inventories to the five-year average - are we above or below?"
3. "Calculate the average refinery utilization rate mentioned in recent reports"

**Expected Agent Steps:**
1. Search for relevant data (RAG tool)
2. Extract specific numbers (data extractor tool)
3. Perform calculation (calculator tool)
4. Formulate answer

**Requirements:**
- Choose or create a multi-step question
- Run the agent
- Verify the agent used multiple tools
- Check if the answer is correct

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if agent_executor:
    complex_question = """
    Find the crude oil inventory level from the most recent report and from one month ago.
    Calculate the percent change. Is this bullish or bearish?
    """
    
    print("Complex Multi-Step Question:")
    print("="*70)
    print(complex_question)
    print("\nAgent Execution:")
    print("-"*70)
    
    try:
        result = agent_executor.invoke({"input": complex_question})
        
        print("\n" + "="*70)
        print("Final Answer:")
        print("="*70)
        print(result['output'])
        
        print("\n" + "="*70)
        print("Analysis:")
        print("="*70)
        print("The agent should have:")
        print("  1. Used eia_report_search to find recent inventory data")
        print("  2. Used eia_report_search with date filter for older data")
        print("  3. Used data_extractor to get specific numbers")
        print("  4. Used calculator to compute percent change")
        print("  5. Interpreted result (decrease = bullish, increase = bearish)")
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
else:
    print("⚠️  Agent not available")

## 4. Advanced Agent Features

Production research agents often include:

1. **Memory** - Remember context from previous questions
2. **Planning** - Break complex questions into subtasks
3. **Self-Correction** - Retry failed actions
4. **Source Citation** - Track which documents were used
5. **Confidence Scores** - Indicate certainty

We'll add conversation memory to enable follow-up questions.

In [ ]:
# Purpose: Create agent with conversation memory
# Key Concept: Memory enables multi-turn research sessions

if llm and tools:
    # Create memory
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    
    # Update prompt to include chat history
    react_prompt_with_memory = PromptTemplate.from_template("""
You are a commodity market research analyst with access to EIA petroleum reports and analysis tools.

Previous conversation:
{chat_history}

Answer the following question by using the available tools. Think step by step.

You have access to these tools:
{tools}

Tool names: {tool_names}

Use this format:

Question: the input question
Thought: think about what to do
Action: tool name
Action Input: the input to the tool
Observation: the result of the action
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")
    
    # Create agent with memory
    agent_with_memory = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=react_prompt_with_memory
    )
    
    agent_executor_with_memory = AgentExecutor(
        agent=agent_with_memory,
        tools=tools,
        memory=memory,
        verbose=True,
        max_iterations=10,
        handle_parsing_errors=True
    )
    
    print("✅ Research agent with memory created")
else:
    agent_executor_with_memory = None
    print("⚠️  Cannot create agent with memory")

In [ ]:
# Purpose: Test multi-turn conversation with memory
# Key Concept: Agent remembers context for follow-up questions

if agent_executor_with_memory:
    print("Multi-Turn Research Session\n")
    print("="*70)
    
    questions = [
        "What were crude oil inventories in the most recent EIA report?",
        "How does that compare to the five-year average?",
        "Is this bullish or bearish for prices?"
    ]
    
    for i, question in enumerate(questions, 1):
        print(f"\n[Question {i}] {question}")
        print("-"*70)
        
        try:
            result = agent_executor_with_memory.invoke({"input": question})
            print(f"\n[Answer {i}] {result['output']}")
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
    
    print("\n" + "="*70)
    print("Conversation Complete")
    print("="*70)
    print("Note: The agent used context from previous answers to handle follow-ups")
else:
    print("⚠️  Agent with memory not available")

### 💡 Exercise 4.1: Build a Custom Tool

**Task:** Create a new tool for the agent and test it.

**Tool Ideas:**
1. **Price Data Tool** - Fetch historical commodity prices from an API
2. **Comparison Tool** - Compare current data to historical averages
3. **Trend Analyzer** - Identify if metric is rising/falling/stable
4. **Unit Converter** - Convert between barrels, gallons, metric tons, etc.

**Requirements:**
- Create a new tool class inheriting from `BaseTool`
- Define name, description, and args_schema
- Implement `_run()` method
- Add tool to agent's tool list
- Test with a question that requires the new tool

**Hint:** Follow the pattern from `CalculatorTool` and `DataExtractorTool`

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION: Unit Converter Tool
# ------------------------------

class UnitConverterInput(BaseModel):
    """Input for unit converter."""
    value: float = Field(description="Numerical value to convert")
    from_unit: str = Field(description="Source unit: 'barrels', 'gallons', 'metric_tons', 'liters'")
    to_unit: str = Field(description="Target unit: 'barrels', 'gallons', 'metric_tons', 'liters'")

class UnitConverterTool(BaseTool):
    """Tool for converting between commodity units."""
    
    name = "unit_converter"
    description = """
    Convert between common commodity units:
    - barrels (bbl)
    - gallons (gal)
    - metric tons (mt)
    - liters (L)
    
    Example: Convert 1000 barrels to gallons
    Input: value=1000, from_unit='barrels', to_unit='gallons'
    Output: 42000 gallons
    """
    args_schema: type[BaseModel] = UnitConverterInput
    
    def _run(self, value: float, from_unit: str, to_unit: str) -> str:
        """Convert between units."""
        # Conversion factors to barrels
        to_barrels = {
            'barrels': 1.0,
            'gallons': 1.0 / 42.0,
            'liters': 1.0 / 158.987,
            'metric_tons': 1.0 / 0.136  # Approximate for crude oil
        }
        
        if from_unit not in to_barrels or to_unit not in to_barrels:
            return f"Error: Unknown unit. Supported: {list(to_barrels.keys())}"
        
        # Convert to barrels, then to target unit
        barrels = value * to_barrels[from_unit]
        result = barrels / to_barrels[to_unit]
        
        return f"{result:.2f} {to_unit}"
    
    async def _arun(self, *args, **kwargs):
        return self._run(*args, **kwargs)

# Test the tool
converter_tool = UnitConverterTool()
test_result = converter_tool._run(1000, 'barrels', 'gallons')
print(f"Unit Converter Test: 1000 barrels = {test_result}")

# Add to agent
if llm and tools:
    enhanced_tools = tools + [converter_tool]
    
    agent_enhanced = create_react_agent(
        llm=llm,
        tools=enhanced_tools,
        prompt=react_prompt
    )
    
    agent_executor_enhanced = AgentExecutor(
        agent=agent_enhanced,
        tools=enhanced_tools,
        verbose=True,
        max_iterations=10,
        handle_parsing_errors=True
    )
    
    print("\n✅ Enhanced agent with unit converter created")
    
    # Test with conversion question
    test_question = "If crude oil inventories are 450 million barrels, how many gallons is that?"
    print(f"\nTest Question: {test_question}")
    
    try:
        result = agent_executor_enhanced.invoke({"input": test_question})
        print(f"\nAnswer: {result['output']}")
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
else:
    print("⚠️  Cannot create enhanced agent")

## 5. Production Considerations

When deploying research agents:

**Performance:**
- Cache tool results to avoid redundant calls
- Set timeouts on tool execution
- Limit max iterations to control cost

**Reliability:**
- Handle tool failures gracefully
- Validate tool outputs before using
- Log agent reasoning for debugging

**Safety:**
- Restrict tool access (read-only tools preferred)
- Validate user inputs to prevent injection
- Monitor for unexpected agent behavior

**Cost:**
- Each agent iteration = LLM API call
- Complex questions can use many tokens
- Consider cheaper models for simple tools

### 💡 Exercise 5.1: Research Session Wrapper

**Task:** Create a user-friendly wrapper for the research agent.

**Requirements:**
1. Function that takes a question and returns answer
2. Include metadata: tools used, iterations, tokens consumed
3. Format output nicely
4. Handle errors with helpful messages
5. Optional: Add logging of agent steps

**Signature:**
```python
def research_query(question: str, verbose: bool = False) -> Dict[str, Any]:
    """
    Run research agent on question.
    
    Returns:
    --------
    dict with keys: answer, tools_used, iterations, success, error (if any)
    """
```

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

def research_query(question: str, verbose: bool = False) -> Dict[str, Any]:
    """
    Run research agent on question with error handling and metadata.
    
    Parameters:
    -----------
    question : str
        Research question
    verbose : bool
        Show agent reasoning steps
        
    Returns:
    --------
    dict
        Result with answer and metadata
    """
    if not agent_executor:
        return {
            'success': False,
            'error': 'Research agent not available',
            'answer': None
        }
    
    start_time = datetime.now()
    
    try:
        # Run agent
        result = agent_executor.invoke({"input": question})
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        return {
            'success': True,
            'answer': result['output'],
            'duration_seconds': duration,
            'question': question,
            'timestamp': end_time.isoformat(),
            'error': None
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'answer': None,
            'question': question
        }

# Test the wrapper
test_questions = [
    "What is the current level of crude oil inventories?",
    "Convert 500 million barrels to gallons"
]

print("Research Query Wrapper Test\n")
print("="*70)

for question in test_questions:
    print(f"\nQ: {question}")
    print("-"*70)
    
    result = research_query(question, verbose=False)
    
    if result['success']:
        print(f"✅ Answer: {result['answer']}")
        print(f"   Duration: {result['duration_seconds']:.2f}s")
    else:
        print(f"❌ Error: {result['error']}")

print("\n" + "="*70)

## Summary

### Key Takeaways

1. **Agents Extend RAG** - Combine retrieval with reasoning and computation

2. **ReAct Pattern Works** - Thought→Action→Observation loop is effective

3. **Tools Are Modular** - Easy to add domain-specific capabilities

4. **Memory Enables Conversation** - Multi-turn research sessions are natural

5. **Production Requires Guardrails** - Timeouts, validation, error handling essential

### Skills Gained

✅ Building custom tools for LLM agents  
✅ Implementing ReAct pattern with LangChain  
✅ Creating multi-step reasoning workflows  
✅ Adding conversation memory  
✅ Testing and debugging agent behavior  
✅ Production-ready agent wrappers  

### What's Next

In **Module 2, Notebook 3** (`03_query_optimization.ipynb`), we'll learn:
- Query rewriting for better retrieval
- HyDE (Hypothetical Document Embeddings)
- Self-query retrieval with metadata filters
- Optimizing search relevance

### Additional Resources

- **LangChain Agents:** https://python.langchain.com/docs/modules/agents/
- **ReAct Paper:** https://arxiv.org/abs/2210.03629
- **Tool Use Best Practices:** Anthropic and OpenAI documentation

---

**Next:** [Module 2, Notebook 3 - Query Optimization](03_query_optimization.ipynb)